# ComfyUI on free Google Colab (T4)

Minimal, current notebook: clones the latest ComfyUI, installs its requirements, downloads the Stable Diffusion 1.5 checkpoint and launches ComfyUI behind a free Cloudflare tunnel. Run the cells in order, top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!python3 --version

In [ ]:
!rm -rf /content/ComfyUI /content/cloudflared
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt
!git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!ls /content/ComfyUI/main.py && echo INSTALL-OK

In [ ]:
!mkdir -p /content/ComfyUI/models/checkpoints
!wget -O /content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly.safetensors https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
!ls -lh /content/ComfyUI/models/checkpoints/

In [ ]:
import re, subprocess, threading
from queue import Queue

def start_tunnel(port, queue):
    process = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            queue.put(match.group(0))
            break
    process.wait()

queue = Queue()
thread = threading.Thread(target=start_tunnel, args=(8188, queue), daemon=True)
thread.start()
print('Waiting for the Cloudflare tunnel...')
tunnel_url = queue.get(timeout=180)
print('ComfyUI public link:', tunnel_url)

!python /content/ComfyUI/main.py --dont-print-server --listen 127.0.0.1 --port 8188